# **read data**

In [1]:
import numpy as np
import pandas as pd
import sys

In [2]:
data = pd.read_csv("/content/Result_14.csv")

In [3]:
data = data.rename(columns = {"WEEK_NUMBER": "WEEK", "H2": "TIME RANGE"})
data.head(2)

,YEAR,WEEK,TIME RANGE,LEG,DEPTIME,ARRTIME,DEPDATE,LF
0,2026,34,10 - 12,ICNDAD,1105,1350,2026-08-18,51
1,2027,9,08 - 10,PUSCXR,805,1055,2027-03-04,0


In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152242 entries, 0 to 152241
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   YEAR        152242 non-null  int64 
 1   WEEK        152242 non-null  int64 
 2   TIME RANGE  152242 non-null  object
 3   LEG         152242 non-null  object
 4   DEPTIME     152242 non-null  int64 
 5   ARRTIME     152242 non-null  int64 
 6   DEPDATE     152242 non-null  object
 7   LF          152242 non-null  int64 
dtypes: int64(5), object(3)
memory usage: 9.3+ MB


# **function**

In [5]:
# ham chuyen doi hhmm sang tong so phut
def time_to_minutes(time):
    hours = int(time[:2])
    minutes = int(time[2:])
    return hours * 60 + minutes

In [6]:
# ham chuyen so phut sang format hhmm
def minutes_to_time(minute):
    hours = int(minute // 60)
    minutes = int(minute % 60)
    return f"{hours:02d}{minutes:02d}"

# **ham nhan input tu nguoi dung**

In [17]:
def input_info(data):
  print("Type 'quit' or 'exit' at any prompt to stop the program.")

  # nhan thong tin year, leg, loai hanh trinh tu nguoi dung, bao loi va dung chuong trinh neu thong tin nhap vao khong hop le
  # nhap thong tin year
  year_str = input("Enter year: ")
  if year_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  try:
    year = int(year_str)
  except ValueError:
    print(f"Error: Year '{year_str}' is not a valid number.")
    return None

  # nhap thong tin loai hanh trinh, chi chap nhan 2 gia tri inbound/outbound
  while True:
    leg_type = input("Enter leg type (inbound/outbound): ")
    if leg_type.lower() in ['quit', 'exit']:
      print("Exiting program.")
      return None
    if leg_type.lower() in ['inbound', 'outbound']:
      break
    else:
      print("Leg type must be 'inbound' or 'outbound'. Please try again.")

  # nhap thong tin leg
  leg = input("Enter leg: ")
  if leg.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None

  # print thong tin khung gio cua leg, year vua nhap
  time_range_options = data[(data['YEAR'] == year) & (data['LEG'] == leg)]['TIME RANGE'].unique()
  if len(time_range_options) == 0:
    print(f"Error: No data found for Leg '{leg}', Year {year}.")
    return None
  print(f"Time range for leg '{leg}' of year {year}: {np.sort(time_range_options)}")

  # nhap thong tin time range
  time_range_str = input("Enter time range: ")
  if time_range_str.lower() in ['quit', 'exit']:
    print("Exiting program.")
    return None
  time_range = time_range_str

  # filter the data and return the required columns as a DataFrame
  filtered_data = data[(data['YEAR'] == year) & (data['LEG'] == leg) & (data['TIME RANGE'] == time_range)]
  # raise error if filtered_data empty
  if filtered_data.empty:
    print(f"Error: No data found for Leg '{leg}', Year {year}, Time Range '{time_range}'.")
    return None

  # select and return the desired columns, using existing 'LF'
  return_df = filtered_data[['YEAR', 'WEEK', 'LEG', 'DEPTIME', 'ARRTIME', 'TIME RANGE', 'LF']].copy()

  # avg arr/dep time based on leg_type
  if leg_type.lower() == 'inbound':
    return_df['Time_Str'] = return_df['ARRTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgArrTime'
  else:
    return_df['Time_Str'] = return_df['DEPTIME'].astype(str).str.zfill(4)
    return_df['Time_Mean'] = return_df['Time_Str'].apply(time_to_minutes)
    time_column_name_for_output = 'AvgDepTime'

  # groupby and calculate mean for LF and Avg_Minutes
  result = return_df.groupby(['YEAR', 'WEEK', 'LEG', 'TIME RANGE']).agg(LF = ('LF', 'mean'), Avg_Minutes = ('Time_Mean', 'mean')).reset_index()

  # tinh avg time
  result[time_column_name_for_output] = result['Avg_Minutes'].apply(minutes_to_time)

  # chon cot hien thi o output
  input_df = result[['YEAR', 'WEEK', 'LEG', 'TIME RANGE', 'LF', time_column_name_for_output]]
  #print(input_df)
  return input_df, leg_type

# **ham thuc thi neu nguoi dung chon loai hanh trinh inbound**

In [18]:
def inbound(leg_input, year, time_range):
  outbound_legs_data = []
  target_arrival_airport = leg_input[-3:]

  # input data (no longer filtering by specific week, gets all weeks for the selected year, leg, time_range)
  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIME RANGE'] == time_range)].copy()

  # format arrival time cua leg input
  original_leg_data['Time_Str'] = original_leg_data['ARRTIME'].astype(str).str.zfill(4)
  original_leg_data['Time_Min'] = original_leg_data['Time_Str'].apply(time_to_minutes)
  # tinh avg arrival time PER WEEK
  original_leg_weekly_arrival_time = original_leg_data.groupby('WEEK')['Time_Min'].mean().reset_index()
  original_leg_weekly_arrival_time = original_leg_weekly_arrival_time.rename(columns={'Time_Min': 'original_leg_avg_arrival_time'})

  # filter year (already contains all weeks)
  filtered_context_data = data[(data['YEAR'] == year)].copy()

  # filter cac leg co 3 ki tu dau giong 3 ki tu cuoi cua leg input
  potential_outbound_legs = filtered_context_data[filtered_context_data['LEG'].str.startswith(target_arrival_airport)].copy()

  if potential_outbound_legs.empty:
    return pd.DataFrame(columns=['YEAR', 'WEEK', 'LEG', 'TIME RANGE', 'LF', 'DURATION', 'DEPTIME'])

  # deptime theo tong so phut
  potential_outbound_legs['DEPTIME_Str'] = potential_outbound_legs['DEPTIME'].astype(str).str.zfill(4)
  potential_outbound_legs['DEPTIME_Min'] = potential_outbound_legs['DEPTIME_Str'].apply(time_to_minutes)

  # group by leg, time range, WEEK de tinh LF, avg deptime
  grouped_outbound = potential_outbound_legs.groupby(['YEAR', 'WEEK', 'LEG', 'TIME RANGE']).agg(LF = ('LF', 'mean'), avg_deptime_min = ('DEPTIME_Min', 'mean')).reset_index()

  # tinh duration = avg deptime - avg arrival time (now merged by week)
  grouped_outbound = pd.merge(grouped_outbound, original_leg_weekly_arrival_time, on='WEEK', how='left')
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['avg_deptime_min'] - grouped_outbound['original_leg_avg_arrival_time']
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['DURATION_in_minutes'].apply(lambda x: x + 24 * 60 if x < 0 else x)
  grouped_outbound['DURATION_in_minutes'] = grouped_outbound['DURATION_in_minutes'].fillna(0) # fill NaN with 0
  grouped_outbound['DURATION'] = grouped_outbound['DURATION_in_minutes'].apply(minutes_to_time)

  # quy doi avg deptime ve format hhmm
  grouped_outbound['DEPTIME'] = grouped_outbound['avg_deptime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_outbound[['YEAR', 'WEEK', 'LEG', 'TIME RANGE', 'LF', 'DURATION', 'DEPTIME']].copy()
  result_df = result_df.sort_values(by = ['YEAR', 'WEEK', 'LEG', 'TIME RANGE'], ascending = True).reset_index(drop = True)

  return result_df

# **ham thuc thi neu nguoi dung chon loai hanh trinh outbound**

In [19]:
def outbound(leg_input, year, time_range):
  target_departure_airport = leg_input[:3]

  # input data (no longer filtering by specific week, gets all weeks for the selected year, leg, time_range)
  original_leg_data = data[(data['YEAR'] == year) & (data['LEG'] == leg_input) & (data['TIME RANGE'] == time_range)].copy()

  original_leg_data['Time_Str'] = original_leg_data['DEPTIME'].astype(str).str.zfill(4)
  original_leg_data['Time_Min'] = original_leg_data['Time_Str'].apply(time_to_minutes)
  # tinh avg deptime cua leg input PER WEEK
  original_leg_weekly_deptime = original_leg_data.groupby('WEEK')['Time_Min'].mean().reset_index()
  original_leg_weekly_deptime = original_leg_weekly_deptime.rename(columns={'Time_Min': 'original_leg_avg_deptime'})

  # filter year (already contains all weeks)
  filtered_context_data = data[(data['YEAR'] == year)].copy()

  potential_inbound_legs = filtered_context_data[filtered_context_data['LEG'].str.endswith(target_departure_airport)].copy()

  if potential_inbound_legs.empty:
    return pd.DataFrame(columns = ['YEAR', 'WEEK', 'LEG', 'TIME RANGE', 'LF', 'DURATION', 'ARRTIME'])

  potential_inbound_legs['ARRTIME_Str'] = potential_inbound_legs['ARRTIME'].astype(str).str.zfill(4)
  potential_inbound_legs['ARRTIME_Min'] = potential_inbound_legs['ARRTIME_Str'].apply(time_to_minutes)

  # group by theo leg, time range, WEEK de tinh total bkg, cap, LF
  grouped_inbound = potential_inbound_legs.groupby(['YEAR', 'WEEK', 'LEG', 'TIME RANGE']).agg(LF = ('LF', 'mean'), avg_arrtime_min = ('ARRTIME_Min', 'mean')).reset_index()

  # tinh duration = avg deptime leg input - avg arrival time current leg
  grouped_inbound = pd.merge(grouped_inbound, original_leg_weekly_deptime, on='WEEK', how='left')
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['original_leg_avg_deptime'] - grouped_inbound['avg_arrtime_min']
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['DURATION_in_minutes'].apply(lambda x: x + 24 * 60 if x < 0 else x)
  grouped_inbound['DURATION_in_minutes'] = grouped_inbound['DURATION_in_minutes'].fillna(0)
  grouped_inbound['DURATION'] = grouped_inbound['DURATION_in_minutes'].apply(minutes_to_time)

  # quy doi arrival time ve format hhmm
  grouped_inbound['ARRTIME'] = grouped_inbound['avg_arrtime_min'].apply(minutes_to_time)

  # output
  result_df = grouped_inbound[['YEAR', 'WEEK', 'LEG', 'TIME RANGE', 'LF', 'DURATION', 'ARRTIME']].copy()
  result_df = result_df.sort_values(by=['YEAR', 'WEEK', 'LEG', 'TIME RANGE'], ascending=True).reset_index(drop=True)

  return result_df

# **output**

In [20]:
# goi ham input_info khi nguoi dung nhap thong tin
user_inputs = input_info(data)

if user_inputs is not None:
  df_input_info, leg_type_input = user_inputs
  leg_input = df_input_info['LEG'].iloc[0]
  year_input = df_input_info['YEAR'].iloc[0]
  time_range_input = df_input_info['TIME RANGE'].iloc[0]

  print("User Input Details:")
  display(df_input_info)

  # goi ham inbound/outbound tuy theo loai hanh trinh nguoi dung nhap
  if leg_type_input.lower() == 'inbound':
    result_df = inbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nInbound Legs:")
      display(result_df)
    else:
      print("No inbound legs found for the given input.")
  elif leg_type_input.lower() == 'outbound':
    result_df = outbound(leg_input, year_input, time_range_input)
    if not result_df.empty:
      print("\nOutbound Legs:")
      display(result_df)
    else:
      print("No outbound legs found for the given input.")
else:
  print("Input process terminated or invalid input provided. No further processing in this cell.")

Type 'quit' or 'exit' at any prompt to stop the program.
Enter year: 2026
Enter leg type (inbound/outbound): inbound
Enter leg: BKKHAN
Time range for leg 'BKKHAN' of year 2026: ['10 - 12' '12 - 14' '14 - 16' '18 - 20' '22 - 24']
Enter time range: 14 - 16
User Input Details:


,YEAR,WEEK,LEG,TIME RANGE,LF,AvgArrTime
0,2026,11,BKKHAN,14 - 16,99.166667,1750
1,2026,12,BKKHAN,14 - 16,92.000000,1750
2,2026,13,BKKHAN,14 - 16,87.857143,1750
3,2026,14,BKKHAN,14 - 16,84.857143,1755
4,2026,15,BKKHAN,14 - 16,77.714286,1755
5,2026,16,BKKHAN,14 - 16,79.285714,1755
6,2026,17,BKKHAN,14 - 16,60.428571,1755
7,2026,18,BKKHAN,14 - 16,63.428571,1755
8,2026,19,BKKHAN,14 - 16,38.714286,1755
9,2026,20,BKKHAN,14 - 16,37.142857,1755



Inbound Legs:


,YEAR,WEEK,LEG,TIME RANGE,LF,DURATION,DEPTIME
0,2026,11,HANBKK,08 - 10,89.333333,1525,0915
1,2026,11,HANBKK,12 - 14,87.833333,1850,1240
2,2026,11,HANBKK,14 - 16,89.800000,2202,1552
3,2026,11,HANBKK,16 - 18,91.000000,2222,1612
4,2026,11,HANBKK,18 - 20,91.000000,0110,1900
...,...,...,...,...,...,...,...
3650,2026,53,HANVCL,06 - 08,0.000000,1325,0715
3651,2026,53,HANVDH,08 - 10,0.000000,1455,0845
3652,2026,53,HANVII,06 - 08,0.000000,1315,0705
3653,2026,53,HANVII,16 - 18,0.250000,2300,1650


In [21]:
df_input_info = df_input_info.pivot(index = ['YEAR', 'LEG', 'TIME RANGE'], columns = 'WEEK', values = ['LF']).reset_index()
df_input_info

YEAR     LEG TIME RANGE         LF                              \
WEEK                                  11    12         13         14   
0     2026  BKKHAN    14 - 16  99.166667  92.0  87.857143  84.857143   

                                       ...                                     \
WEEK         15         16         17  ...        44        45        46   47   
0     77.714286  79.285714  60.428571  ...  1.142857  2.571429  0.714286  2.0   

                                                              
WEEK        48        49        50        51        52    53  
0     1.571429  2.714286  0.714286  1.857143  3.714286  2.25  

[1 rows x 46 columns]

In [ ]:
# export df_input_info ra file csv
df_input_info.to_csv('input_info.csv', index=False)
# download file csv
import os
full_path = os.path.abspath('input_info.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\input_info.csv


In [22]:
result_df = result_df.pivot(index = ['LEG', 'TIME RANGE', 'DEPTIME', 'DURATION'], columns = 'WEEK', values = ['LF']).reset_index()
result_df

LEG TIME RANGE DEPTIME DURATION         LF                        \
WEEK                                             11         12         13   
0     HANAMS    02 - 04    0350     0955        NaN        NaN        NaN   
1     HANBKK    08 - 10    0850     1455        NaN        NaN        NaN   
2     HANBKK    08 - 10    0853     1459        NaN        NaN        NaN   
3     HANBKK    08 - 10    0911     1520        NaN        NaN  80.142857   
4     HANBKK    08 - 10    0915     1525  89.333333  83.142857        NaN   
..       ...        ...     ...      ...        ...        ...        ...   
392   HANVII    16 - 18    1720     2325        NaN        NaN        NaN   
393   HANVTE    08 - 10    0920     1525        NaN        NaN        NaN   
394   HANVTE    08 - 10    0922     1527        NaN        NaN        NaN   
395   HANVTE    08 - 10    0932     1542        NaN        NaN  36.428571   
396   HANVTE    08 - 10    0935     1545  85.833333  59.428571        NaN   

                                       ...                              \
WEEK         14         15         16  ...    44         45         46   
0           NaN        NaN        NaN  ...   NaN        NaN        NaN   
1     65.571429  67.000000  50.857143  ...   NaN        NaN        NaN   
2           NaN        NaN        NaN  ...   NaN        NaN        NaN   
3           NaN        NaN        NaN  ...   NaN        NaN        NaN   
4           NaN        NaN        NaN  ...   1.0   1.428571   2.571429   
..          ...        ...        ...  ...   ...        ...        ...   
392   18.285714  13.428571  13.714286  ...   NaN        NaN        NaN   
393   28.000000  28.714286  33.428571  ...   NaN        NaN        NaN   
394         NaN        NaN        NaN  ...   NaN        NaN        NaN   
395         NaN        NaN        NaN  ...   NaN        NaN        NaN   
396         NaN        NaN        NaN  ...  15.0  19.000000  21.428571   

                                                                             
WEEK         47         48         49         50         51        52    53  
0           NaN        NaN        NaN        NaN        NaN       NaN   NaN  
1           NaN        NaN        NaN        NaN        NaN       NaN   NaN  
2           NaN        NaN        NaN        NaN        NaN       NaN   NaN  
3           NaN        NaN        NaN        NaN        NaN       NaN   NaN  
4      1.857143   1.571429   3.285714  10.285714   9.000000  6.571429  9.25  
..          ...        ...        ...        ...        ...       ...   ...  
392         NaN        NaN        NaN        NaN        NaN       NaN   NaN  
393         NaN        NaN        NaN        NaN        NaN       NaN   NaN  
394         NaN        NaN        NaN        NaN        NaN       NaN   NaN  
395         NaN        NaN        NaN        NaN        NaN       NaN   NaN  
396   29.857143  20.857143  17.285714  12.571429  14.142857  6.428571  2.00  

[397 rows x 47 columns]

In [ ]:
# export result_df ra file csv
result_df.to_csv('result_df.csv', index=False)
# download file csv
import os
full_path = os.path.abspath('result_df.csv')
print(f"File path: {full_path}")

File path: c:\Users\hiennguyenthithanh\Downloads\result_df.csv
